#**Connect Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


#**Load Dataset**

In [4]:
from pathlib import Path

DATA_DIR = Path("/content/APTOS")
CSV_PATH = DATA_DIR / "train.csv"
IMAGE_DIR = DATA_DIR / "train_images"

# Check whether dataset is already fully available
image_count = len(list(IMAGE_DIR.glob("*.png"))) if IMAGE_DIR.exists() else 0

if CSV_PATH.exists() and image_count == 3662:
    print("APTOS already ready.")
    print("CSV:", CSV_PATH)
    print("Images:", image_count)
    print("Skipping download and unzip.")

else:
    print("APTOS not complete. Downloading...")

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

    # Download only if ZIP does not already exist
    if not (DATA_DIR / "aptos2019-blindness-detection.zip").exists():
        !kaggle competitions download -c aptos2019-blindness-detection -p /content/APTOS

    # -o = overwrite automatically, NO questions
    !unzip -q -o /content/APTOS/aptos2019-blindness-detection.zip -d /content/APTOS

    image_count = len(list(IMAGE_DIR.glob("*.png")))

    print("CSV:", CSV_PATH.exists())
    print("Images:", image_count)

APTOS not complete. Downloading...
100% 9.51G/9.51G [07:56<00:00, 21.4MB/s]

CSV: True
Images: 3662


#**MedSigLIP APTOS embedding extraction**

**Loading from Hugging Face**

In [8]:
from huggingface_hub import login
login()

In [9]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("google/medsiglip-448")

print("ACCESS OK")

preprocessor_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  798kB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

ACCESS OK


In [10]:
# ============================================================
# APTOS -> MedSigLIP Frozen Embeddings
#
# CONSISTENT WITH ORIGINAL RETFOUND PIPELINE
#
# Same:
#   - raw APTOS images
#   - duplicate audit
#   - 3504 KEEP cases
#   - black-border removal
#   - square padding
#
# Different ONLY where model-specific:
#   - MedSigLIP native processor / input resolution
#
# Saves ONLY:
#   aptos_medsiglip_embeddings.npz
# ============================================================

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel


# ============================================================
# 1. PATHS
# ============================================================

IMAGE_DIR = Path(
    "/content/APTOS/train_images"
)

AUDIT_PATH = Path(
    "/content/drive/MyDrive/thesis_DR/aptos_exact_duplicate_audit.csv"
)

SAVE_PATH = Path(
    "/content/drive/MyDrive/thesis_DR/aptos_medsiglip_embeddings.npz"
)

MODEL_ID = "google/medsiglip-448"


# ============================================================
# 2. LOAD EXACT SAME 3504 KEEP IMAGES AS RETFOUND
# ============================================================

audit = pd.read_csv(
    AUDIT_PATH,
    dtype={"image_id": str}
)

keep_df = (
    audit[
        audit["action"] == "KEEP"
    ][
        [
            "image_id",
            "original_grade",
            "binary_label"
        ]
    ]
    .sort_values("image_id")
    .reset_index(drop=True)
)

print("APTOS KEEP images:", len(keep_df))

assert len(keep_df) == 3504


# Confirm raw images exist
missing = [
    image_id
    for image_id in keep_df["image_id"]
    if not (
        IMAGE_DIR /
        f"{image_id}.png"
    ).exists()
]

assert len(missing) == 0, (
    f"{len(missing)} raw APTOS images missing. "
    "Rerun the original RETFound Load Dataset cell."
)

print("Raw image check: PASS")


# ============================================================
# 3. EXACT SAME PREPROCESSING BEFORE MODEL-SPECIFIC PROCESSOR
# ============================================================

def remove_black_border(
    image,
    threshold=7
):

    img = np.asarray(image)

    gray = (
        0.299 * img[:, :, 0]
        +
        0.587 * img[:, :, 1]
        +
        0.114 * img[:, :, 2]
    )

    mask = gray > threshold

    if not mask.any():
        return image

    rows = np.where(
        mask.any(axis=1)
    )[0]

    cols = np.where(
        mask.any(axis=0)
    )[0]

    cropped = img[
        rows[0]:rows[-1] + 1,
        cols[0]:cols[-1] + 1
    ]

    return Image.fromarray(
        cropped
    )


def pad_to_square(image):

    width, height = image.size

    side = max(
        width,
        height
    )

    square = Image.new(
        "RGB",
        (side, side),
        (0, 0, 0)
    )

    left = (
        side - width
    ) // 2

    top = (
        side - height
    ) // 2

    square.paste(
        image,
        (left, top)
    )

    return square


# ============================================================
# 4. DATASET
# ============================================================

class APTOSMedSigLIPDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir
    ):

        self.df = dataframe
        self.image_dir = image_dir


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image_id = str(
            row["image_id"]
        )

        image_path = (
            self.image_dir /
            f"{image_id}.png"
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        # EXACT SAME custom preprocessing as RETFound
        image = remove_black_border(
            image
        )

        image = pad_to_square(
            image
        )

        return (
            image,
            image_id,
            int(
                row["original_grade"]
            ),
            int(
                row["binary_label"]
            )
        )


# ============================================================
# 5. LOAD MEDSIGLIP
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


processor = (
    AutoProcessor.from_pretrained(
        MODEL_ID
    )
)


model = (
    AutoModel.from_pretrained(
        MODEL_ID
    )
)

# Frozen encoder
for parameter in model.parameters():
    parameter.requires_grad = False

model = model.to(
    device
)

model.eval()


print("MedSigLIP loaded.")

print(
    "Encoder frozen:",
    not any(
        p.requires_grad
        for p in model.parameters()
    )
)


# ============================================================
# 6. DATASET
# ============================================================

dataset = APTOSMedSigLIPDataset(
    keep_df,
    IMAGE_DIR
)


# We keep image loading simple because the Hugging Face
# processor operates on PIL images.
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    collate_fn=lambda batch: batch,
    pin_memory=False
)


# ============================================================
# 7. EXTRACT EMBEDDINGS
# ============================================================

all_embeddings = []

all_image_ids = []

all_grades = []

all_binary_labels = []


with torch.inference_mode():

    for batch in tqdm(
        loader,
        desc="Extracting MedSigLIP embeddings"
    ):

        images = [
            item[0]
            for item in batch
        ]

        image_ids = [
            item[1]
            for item in batch
        ]

        grades = [
            item[2]
            for item in batch
        ]

        labels = [
            item[3]
            for item in batch
        ]


        # MedSigLIP-specific resize + normalization
        inputs = processor(
            images=images,
            return_tensors="pt"
        )

        pixel_values = inputs[
            "pixel_values"
        ].to(device)


        features = (
            model.get_image_features(
                pixel_values=pixel_values
            )
        )


        # Some Transformers versions return an output object.
        if hasattr(
            features,
            "pooler_output"
        ):
            features = (
                features.pooler_output
            )


        features = (
            features
            .float()
            .cpu()
            .numpy()
        )


        all_embeddings.append(
            features
        )

        all_image_ids.extend(
            image_ids
        )

        all_grades.extend(
            grades
        )

        all_binary_labels.extend(
            labels
        )


# ============================================================
# 8. COMBINE
# ============================================================

embeddings = np.concatenate(
    all_embeddings,
    axis=0
).astype(np.float32)


image_ids = np.asarray(
    all_image_ids
)


grades = np.asarray(
    all_grades,
    dtype=np.int64
)


binary_labels = np.asarray(
    all_binary_labels,
    dtype=np.int64
)


# ============================================================
# 9. VERIFY
# ============================================================

assert embeddings.shape[0] == 3504

assert len(
    image_ids
) == 3504

assert len(
    np.unique(image_ids)
) == 3504

assert (
    image_ids.tolist()
    ==
    keep_df[
        "image_id"
    ].astype(str).tolist()
)

assert not np.isnan(
    embeddings
).any()

assert not np.isinf(
    embeddings
).any()


# ============================================================
# 10. SAVE ONE PERMANENT FILE
# ============================================================

np.savez_compressed(
    SAVE_PATH,

    embeddings=embeddings,

    image_id=image_ids,

    original_grade=grades,

    binary_label=binary_labels
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

print(
    "\n========== MEDSIGLIP EMBEDDING EXTRACTION =========="
)

print("\nModel:")
print("MedSigLIP")

print(
    "Encoder status: FROZEN"
)

print("\nShared preprocessing:")
print(
    "1. Remove black outer border, threshold = 7"
)
print(
    "2. Preserve aspect ratio"
)
print(
    "3. Pad to square"
)

print("\nModel-specific preprocessing:")
print(
    "4. MedSigLIP processor resize/normalization"
)

print(
    "\nImages processed:",
    len(image_ids)
)

print(
    "Embedding shape:",
    embeddings.shape
)

print(
    "NaN values:",
    np.isnan(
        embeddings
    ).sum()
)

print(
    "Infinite values:",
    np.isinf(
        embeddings
    ).sum()
)

print("\nSaved permanently:")
print(SAVE_PATH)

print(
    "\n========== COMPLETE =========="
)

APTOS KEEP images: 3504
Raw image check: PASS
Device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.51GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

MedSigLIP loaded.
Encoder frozen: True


Extracting MedSigLIP embeddings:   0%|          | 0/438 [00:00<?, ?it/s]


========== MEDSIGLIP EMBEDDING EXTRACTION ==========

Model:
MedSigLIP
Encoder status: FROZEN

Shared preprocessing:
1. Remove black outer border, threshold = 7
2. Preserve aspect ratio
3. Pad to square

Model-specific preprocessing:
4. MedSigLIP processor resize/normalization

Images processed: 3504
Embedding shape: (3504, 1152)
NaN values: 0
Infinite values: 0

Saved permanently:
/content/drive/MyDrive/thesis_DR/aptos_medsiglip_embeddings.npz

========== COMPLETE ==========


#**MedSigLIP MLP training**

In [12]:

# FINAL MEDSIGLIP MLP TRAINING
#
# - Frozen MedSigLIP embeddings
# - Development split ONLY for HPO/training
# - 3-fold inner stratified CV
# - Small predefined HPO
# - Early stopping, max 25 epochs
# - Final retraining on full development set
# ============================================================

import random
import copy
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

EMBEDDING_PATH = (
    PROJECT_DIR / "aptos_medsiglip_embeddings.npz"
)

SPLIT_PATH = (
    PROJECT_DIR / "aptos_split_manifest.csv"
)

MODEL_SAVE_PATH = (
    PROJECT_DIR / "aptos_medsiglip_mlp_heads.pt"
)

PRED_SAVE_PATH = (
    PROJECT_DIR / "aptos_medsiglip_raw_predictions.csv"
)


# ============================================================
# 2. LOCKED SETTINGS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

INPUT_DIM = 1152
HIDDEN_DIM = 512
DROPOUT = 0.10

BATCH_SIZE = 64

MAX_EPOCHS = 25
EARLY_STOP_PATIENCE = 4
MIN_DELTA = 1e-5

INNER_FOLDS = 3

# Small prespecified HPO only
HPO_CONFIGS = [
    {
        "name": "A",
        "learning_rate": 3e-4,
        "weight_decay": 0.0
    },
    {
        "name": "B",
        "learning_rate": 3e-4,
        "weight_decay": 1e-4
    },
    {
        "name": "C",
        "learning_rate": 1e-3,
        "weight_decay": 0.0
    },
    {
        "name": "D",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4
    }
]


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 4. MODEL
# ============================================================

class MLPHead(nn.Module):

    def __init__(
        self,
        input_dim=1152,
        hidden_dim=512,
        dropout=0.10
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


# ============================================================
# 5. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 6. LOAD EMBEDDINGS
# ============================================================

saved_embeddings = np.load(
    EMBEDDING_PATH,
    allow_pickle=False
)

embeddings = saved_embeddings[
    "embeddings"
].astype(np.float32)

embedding_ids = saved_embeddings[
    "image_id"
].astype(str)

embedding_grades = saved_embeddings[
    "original_grade"
].astype(int)

embedding_labels = saved_embeddings[
    "binary_label"
].astype(int)


# ============================================================
# 7. VERIFY EMBEDDINGS
# ============================================================

assert embeddings.shape == (3504, 1152)
assert len(np.unique(embedding_ids)) == 3504
assert np.isfinite(embeddings).all()

print("\n========== INPUT CHECK ==========")
print("Embedding shape:", embeddings.shape)
print("Unique image IDs:", len(np.unique(embedding_ids)))


# ============================================================
# 8. LOAD SPLIT MANIFEST
# ============================================================

manifest = pd.read_csv(
    SPLIT_PATH,
    dtype={"image_id": str}
)

assert sorted(
    manifest["seed"].unique().tolist()
) == SEEDS

print("Manifest rows:", len(manifest))
print("Seeds:", SEEDS)


# ============================================================
# 9. EMBEDDING LOOKUP
# ============================================================

embedding_meta = pd.DataFrame({
    "image_id": embedding_ids,
    "embedding_index": np.arange(len(embedding_ids)),
    "embedding_grade": embedding_grades,
    "embedding_label": embedding_labels
})


# ============================================================
# 10. TRAIN WITH INNER VALIDATION + EARLY STOPPING
# ============================================================

def train_with_early_stopping(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate,
    weight_decay,
    run_seed
):

    set_seed(run_seed)

    model = MLPHead(
        INPUT_DIM,
        HIDDEN_DIM,
        DROPOUT
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_dataset = TensorDataset(
        torch.tensor(
            X_train,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train,
            dtype=torch.float32
        )
    )

    generator = torch.Generator()
    generator.manual_seed(run_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32,
        device=device
    )

    y_val_tensor = torch.tensor(
        y_val,
        dtype=torch.float32,
        device=device
    )

    best_val_loss = float("inf")
    best_epoch = 1
    best_state = None

    patience_counter = 0

    train_history = []
    val_history = []

    for epoch in range(1, MAX_EPOCHS + 1):

        # ---------------------------
        # TRAIN
        # ---------------------------

        model.train()

        running_loss = 0.0
        n_seen = 0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()
            optimizer.step()

            batch_n = batch_x.size(0)

            running_loss += (
                loss.item() * batch_n
            )

            n_seen += batch_n

        train_loss = running_loss / n_seen

        # ---------------------------
        # VALIDATION
        # ---------------------------

        model.eval()

        with torch.inference_mode():

            val_logits = model(
                X_val_tensor
            )

            val_loss = criterion(
                val_logits,
                y_val_tensor
            ).item()

        train_history.append(
            float(train_loss)
        )

        val_history.append(
            float(val_loss)
        )

        # ---------------------------
        # EARLY STOPPING
        # ---------------------------

        if val_loss < (
            best_val_loss - MIN_DELTA
        ):

            best_val_loss = val_loss
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    # Restore best fold model
    model.load_state_dict(
        best_state
    )

    result = {
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "epochs_run": int(len(train_history)),
        "train_loss_history": train_history,
        "val_loss_history": val_history
    }

    del model
    del X_val_tensor
    del y_val_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 11. FINAL TRAINING ON FULL DEVELOPMENT SET
# ============================================================

def train_final_head(
    X,
    y,
    learning_rate,
    weight_decay,
    epochs,
    run_seed
):

    set_seed(run_seed)

    model = MLPHead(
        INPUT_DIM,
        HIDDEN_DIM,
        DROPOUT
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    dataset = TensorDataset(
        torch.tensor(
            X,
            dtype=torch.float32
        ),
        torch.tensor(
            y,
            dtype=torch.float32
        )
    )

    generator = torch.Generator()
    generator.manual_seed(run_seed)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    loss_history = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()
            optimizer.step()

            batch_n = batch_x.size(0)

            running_loss += (
                loss.item() * batch_n
            )

            n_seen += batch_n

        epoch_loss = (
            running_loss / n_seen
        )

        loss_history.append(
            float(epoch_loss)
        )

    return model, loss_history


# ============================================================
# 12. STORAGE
# ============================================================

saved_heads = {}
prediction_tables = []


# ============================================================
# 13. RUN EACH OUTER SEED
# ============================================================

print(
    "\n========== FINAL MEDSIGLIP MLP TRAINING =========="
)

print("\nBackbone: MedSigLIP")
print("Encoder: FROZEN")

print("\nArchitecture:")
print(
    "1152 -> 512 -> ReLU -> "
    "Dropout(0.1) -> 1 logit"
)

print("\nHPO:")
print("3-fold inner stratified CV")
print("Stratification: original grades 0-4")
print("Selection metric: validation BCE loss")

print("\nEarly stopping:")
print("Maximum epochs:", MAX_EPOCHS)
print(
    "Patience:",
    EARLY_STOP_PATIENCE
)

print("\nIMPORTANT:")
print("HPO uses DEVELOPMENT data only.")
print(
    "Probability-calibration data: NOT USED"
)
print(
    "Risk-calibration data: NOT USED"
)
print(
    "Internal-test data: NOT USED"
)


for seed in SEEDS:

    print(
        f"\n\n========== SEED {seed} =========="
    )

    # --------------------------------------------------------
    # Build manifest for this seed
    # --------------------------------------------------------

    seed_df = (
        manifest[
            manifest["seed"] == seed
        ]
        .copy()
        .merge(
            embedding_meta,
            on="image_id",
            how="left",
            validate="one_to_one"
        )
    )

    assert (
        seed_df["embedding_index"]
        .isna()
        .sum()
        == 0
    )

    # Verify labels
    assert (
        seed_df[
            "original_grade"
        ].astype(int).values
        ==
        seed_df[
            "embedding_grade"
        ].astype(int).values
    ).all()

    assert (
        seed_df[
            "binary_label"
        ].astype(int).values
        ==
        seed_df[
            "embedding_label"
        ].astype(int).values
    ).all()

    # --------------------------------------------------------
    # DEVELOPMENT ONLY
    # --------------------------------------------------------

    dev_df = (
        seed_df[
            seed_df["split"]
            == "development"
        ]
        .copy()
        .reset_index(drop=True)
    )

    dev_embedding_indices = (
        dev_df[
            "embedding_index"
        ]
        .astype(int)
        .values
    )

    X_dev = embeddings[
        dev_embedding_indices
    ]

    y_dev = (
        dev_df[
            "binary_label"
        ]
        .astype(int)
        .values
    )

    grades_dev = (
        dev_df[
            "original_grade"
        ]
        .astype(int)
        .values
    )

    print(
        "Development images:",
        len(dev_df)
    )

    print(
        "Non-referable:",
        int((y_dev == 0).sum())
    )

    print(
        "Referable:",
        int((y_dev == 1).sum())
    )

    # --------------------------------------------------------
    # FIXED 3-FOLD INNER CV
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=seed
    )

    hpo_results = []

    # --------------------------------------------------------
    # TEST EACH CONFIGURATION
    # --------------------------------------------------------

    for config_index, config in enumerate(
        HPO_CONFIGS
    ):

        fold_results = []

        # Same folds for every config
        for fold_number, (
            train_idx,
            val_idx
        ) in enumerate(
            skf.split(
                X_dev,
                grades_dev
            ),
            start=1
        ):

            # Same initialization seed across
            # configurations for a given fold
            fold_seed = (
                seed * 100
                + fold_number
            )

            result = train_with_early_stopping(
                X_train=X_dev[train_idx],
                y_train=y_dev[train_idx],

                X_val=X_dev[val_idx],
                y_val=y_dev[val_idx],

                learning_rate=config[
                    "learning_rate"
                ],

                weight_decay=config[
                    "weight_decay"
                ],

                run_seed=fold_seed
            )

            fold_results.append(
                result
            )

        fold_losses = [
            result["best_val_loss"]
            for result in fold_results
        ]

        fold_epochs = [
            result["best_epoch"]
            for result in fold_results
        ]

        mean_val_loss = float(
            np.mean(fold_losses)
        )

        std_val_loss = float(
            np.std(
                fold_losses,
                ddof=1
            )
        )

        config_result = {
            "name": config["name"],
            "learning_rate": config[
                "learning_rate"
            ],
            "weight_decay": config[
                "weight_decay"
            ],
            "mean_val_loss": mean_val_loss,
            "std_val_loss": std_val_loss,
            "fold_best_epochs": fold_epochs,
            "fold_results": fold_results
        }

        hpo_results.append(
            config_result
        )

        print(
            f"\nConfig {config['name']}:"
        )

        print(
            "  Learning rate:",
            config["learning_rate"]
        )

        print(
            "  Weight decay:",
            config["weight_decay"]
        )

        print(
            "  Fold best epochs:",
            fold_epochs
        )

        print(
            "  Mean validation BCE:",
            f"{mean_val_loss:.6f}"
        )

        print(
            "  Std validation BCE:",
            f"{std_val_loss:.6f}"
        )

    # --------------------------------------------------------
    # SELECT BEST CONFIG
    # --------------------------------------------------------

    best_config = min(
        hpo_results,
        key=lambda x: x[
            "mean_val_loss"
        ]
    )

    selected_epochs = int(
        np.median(
            best_config[
                "fold_best_epochs"
            ]
        )
    )

    selected_epochs = max(
        1,
        min(
            selected_epochs,
            MAX_EPOCHS
        )
    )

    print(
        "\n---------- SELECTED ----------"
    )

    print(
        "Configuration:",
        best_config["name"]
    )

    print(
        "Learning rate:",
        best_config[
            "learning_rate"
        ]
    )

    print(
        "Weight decay:",
        best_config[
            "weight_decay"
        ]
    )

    print(
        "Mean CV validation BCE:",
        f"{best_config['mean_val_loss']:.6f}"
    )

    print(
        "Fold best epochs:",
        best_config[
            "fold_best_epochs"
        ]
    )

    print(
        "Final training epochs:",
        selected_epochs
    )

    # --------------------------------------------------------
    # RETRAIN USING ALL 2102 DEVELOPMENT IMAGES
    # --------------------------------------------------------

    final_seed = (
        seed * 1000 + 999
    )

    final_model, final_loss_history = (
        train_final_head(
            X=X_dev,
            y=y_dev,

            learning_rate=best_config[
                "learning_rate"
            ],

            weight_decay=best_config[
                "weight_decay"
            ],

            epochs=selected_epochs,

            run_seed=final_seed
        )
    )

    print(
        "Final development training loss:",
        f"{final_loss_history[-1]:.6f}"
    )

    # --------------------------------------------------------
    # GENERATE RAW PREDICTIONS FOR ALL 3504 IMAGES
    # --------------------------------------------------------

    all_indices = (
        seed_df[
            "embedding_index"
        ]
        .astype(int)
        .values
    )

    X_all = torch.tensor(
        embeddings[
            all_indices
        ],
        dtype=torch.float32,
        device=device
    )

    final_model.eval()

    with torch.inference_mode():

        raw_logits = final_model(
            X_all
        )

        raw_probabilities = (
            torch.sigmoid(
                raw_logits
            )
        )

    raw_logits = (
        raw_logits
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    raw_probabilities = (
        raw_probabilities
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    # --------------------------------------------------------
    # SAVE PREDICTION TABLE IN MEMORY
    # --------------------------------------------------------

    pred_df = seed_df[
        [
            "seed",
            "image_id",
            "split",
            "original_grade",
            "binary_label"
        ]
    ].copy()

    pred_df[
        "raw_logit"
    ] = raw_logits

    pred_df[
        "raw_probability"
    ] = raw_probabilities

    prediction_tables.append(
        pred_df
    )

    # --------------------------------------------------------
    # SAVE MODEL INFORMATION IN MEMORY
    # --------------------------------------------------------

    cpu_state_dict = {
        name: tensor
        .detach()
        .cpu()
        for name, tensor
        in final_model.state_dict().items()
    }

    saved_heads[seed] = {

        "state_dict":
            cpu_state_dict,

        "selected_config":
            best_config["name"],

        "selected_learning_rate":
            float(
                best_config[
                    "learning_rate"
                ]
            ),

        "selected_weight_decay":
            float(
                best_config[
                    "weight_decay"
                ]
            ),

        "selected_epochs":
            int(selected_epochs),

        "development_images":
            int(len(dev_df)),

        "development_non_referable":
            int((y_dev == 0).sum()),

        "development_referable":
            int((y_dev == 1).sum()),

        "final_training_loss":
            float(
                final_loss_history[-1]
            ),

        "final_loss_history":
            final_loss_history,

        "hpo_results":
            hpo_results
    }

    del final_model
    del X_all

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 14. COMBINE ALL RAW PREDICTIONS
# ============================================================

predictions = pd.concat(
    prediction_tables,
    ignore_index=True
)

expected_rows = (
    3504 * len(SEEDS)
)

assert len(predictions) == expected_rows

assert (
    predictions[
        "raw_logit"
    ]
    .isna()
    .sum()
    == 0
)

assert (
    predictions[
        "raw_probability"
    ]
    .isna()
    .sum()
    == 0
)

assert np.isfinite(
    predictions[
        "raw_logit"
    ].values
).all()


# ============================================================
# 15. SAVE FINAL MODEL BUNDLE
# ============================================================

model_bundle = {

    "backbone":
        "MedSigLIP",

    "encoder_status":
        "frozen",

    "embedding_dim":
        1152,

    "head_architecture": {
        "input_dim": 1152,
        "hidden_dim": 512,
        "activation": "ReLU",
        "dropout": 0.10,
        "output_dim": 1
    },

    "training_protocol": {

        "loss":
            "BCEWithLogitsLoss",

        "optimizer":
            "AdamW",

        "batch_size":
            BATCH_SIZE,

        "max_epochs":
            MAX_EPOCHS,

        "early_stopping_patience":
            EARLY_STOP_PATIENCE,

        "inner_cv_folds":
            INNER_FOLDS,

        "inner_cv_stratification":
            "original_grade_0_to_4",

        "hpo_selection_metric":
            "mean_inner_validation_BCE",

        "final_epoch_rule":
            "median_best_epoch_of_selected_configuration",

        "class_weighting":
            None
    },

    "hpo_configs":
        HPO_CONFIGS,

    "seeds":
        SEEDS,

    "heads":
        saved_heads
}

torch.save(
    model_bundle,
    MODEL_SAVE_PATH
)


# ============================================================
# 16. SAVE RAW PREDICTIONS
# ============================================================

predictions.to_csv(
    PRED_SAVE_PATH,
    index=False
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print(
    "\n\n========== FINAL TRAINING SUMMARY =========="
)

print(
    "\nHeads trained:",
    len(saved_heads),
    "/ 5"
)

for seed in SEEDS:

    info = saved_heads[seed]

    print(
        f"\nSeed {seed}:"
    )

    print(
        "  Selected config:",
        info["selected_config"]
    )

    print(
        "  Learning rate:",
        info[
            "selected_learning_rate"
        ]
    )

    print(
        "  Weight decay:",
        info[
            "selected_weight_decay"
        ]
    )

    print(
        "  Final epochs:",
        info[
            "selected_epochs"
        ]
    )

print(
    "\nRaw prediction rows:",
    len(predictions)
)

print(
    "Expected rows:",
    expected_rows
)

print(
    "NaN logits:",
    predictions[
        "raw_logit"
    ]
    .isna()
    .sum()
)

print(
    "Infinite logits:",
    np.isinf(
        predictions[
            "raw_logit"
        ].values
    ).sum()
)

print("\nIMPORTANT:")
print(
    "Probability-calibration data was NOT used for HPO/training."
)

print(
    "Risk-calibration data was NOT used for HPO/training."
)

print(
    "Internal-test data was NOT used for HPO/training."
)

print(
    "Raw probabilities are still UNCALIBRATED."
)

print(
    "No selective threshold has been selected."
)

print(
    "No risk-control method has been applied."
)

print("\nSaved final model bundle:")
print(MODEL_SAVE_PATH)

print("\nSaved raw predictions:")
print(PRED_SAVE_PATH)

print(
    "\n========== COMPLETE =========="
)

Device: cuda

========== INPUT CHECK ==========
Embedding shape: (3504, 1152)
Unique image IDs: 3504
Manifest rows: 17520
Seeds: [42, 43, 44, 45, 46]

========== FINAL MEDSIGLIP MLP TRAINING ==========

Backbone: MedSigLIP
Encoder: FROZEN

Architecture:
1152 -> 512 -> ReLU -> Dropout(0.1) -> 1 logit

HPO:
3-fold inner stratified CV
Stratification: original grades 0-4
Selection metric: validation BCE loss

Early stopping:
Maximum epochs: 25
Patience: 4

IMPORTANT:
HPO uses DEVELOPMENT data only.
Probability-calibration data: NOT USED
Risk-calibration data: NOT USED
Internal-test data: NOT USED


========== SEED 42 ==========
Development images: 2102
Non-referable: 1280
Referable: 822

Config A:
  Learning rate: 0.0003
  Weight decay: 0.0
  Fold best epochs: [12, 19, 25]
  Mean validation BCE: 0.137657
  Std validation BCE: 0.006296

Config B:
  Learning rate: 0.0003
  Weight decay: 0.0001
  Fold best epochs: [12, 19, 25]
  Mean validation BCE: 0.137622
  Std validation BCE: 0.006321

Co

#**MEDSIGLIP PROBABILITY CALIBRATION**

In [13]:
# ============================================================
# MEDSIGLIP PROBABILITY CALIBRATION
# EXACT SAME PROCEDURE AS RETFOUND-GREEN
#
# Uses ONLY:
#   prob_calibration split
#
# Does NOT use:
#   development
#   risk_calibration
#   internal_test
#
# Saves ONLY:
#   aptos_medsiglip_temperature_scaling.json
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy.special import expit


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_PATH = (
    PROJECT_DIR /
    "aptos_medsiglip_raw_predictions.csv"
)

SAVE_PATH = (
    PROJECT_DIR /
    "aptos_medsiglip_temperature_scaling.json"
)


# ============================================================
# 2. SETTINGS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

ECE_BINS = 10


# ============================================================
# 3. LOAD RAW PREDICTIONS
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

print("========== INPUT CHECK ==========")

print("Prediction rows:", len(pred))

print(
    "Seeds:",
    sorted(
        pred["seed"]
        .unique()
        .tolist()
    )
)

assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"]
    .unique()
    .tolist()
) == SEEDS

assert (
    pred["raw_logit"]
    .isna()
    .sum()
    == 0
)

assert np.isfinite(
    pred["raw_logit"].values
).all()


# ============================================================
# 4. METRIC FUNCTIONS
# ============================================================

def binary_nll_from_logits(
    logits,
    labels
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    losses = (
        np.maximum(
            logits,
            0
        )
        -
        logits * labels
        +
        np.log1p(
            np.exp(
                -np.abs(
                    logits
                )
            )
        )
    )

    return float(
        np.mean(
            losses
        )
    )


def brier_score(
    probabilities,
    labels
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    return float(
        np.mean(
            (
                probabilities
                -
                labels
            ) ** 2
        )
    )


def expected_calibration_error(
    probabilities,
    labels,
    n_bins=10
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0
    n = len(labels)


    for i in range(
        n_bins
    ):

        lower = edges[i]
        upper = edges[i + 1]

        if i == n_bins - 1:

            mask = (
                (probabilities >= lower)
                &
                (probabilities <= upper)
            )

        else:

            mask = (
                (probabilities >= lower)
                &
                (probabilities < upper)
            )


        count = int(
            mask.sum()
        )

        if count == 0:
            continue


        confidence = float(
            probabilities[
                mask
            ].mean()
        )

        observed_rate = float(
            labels[
                mask
            ].mean()
        )


        ece += (
            count / n
        ) * abs(
            observed_rate
            -
            confidence
        )


    return float(
        ece
    )


# ============================================================
# 5. FIT ONE TEMPERATURE
# ============================================================

def fit_temperature(
    logits,
    labels
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )


    # Optimize log(T), ensuring T > 0
    def objective(
        log_temperature
    ):

        temperature = np.exp(
            log_temperature
        )

        calibrated_logits = (
            logits
            /
            temperature
        )

        return (
            binary_nll_from_logits(
                calibrated_logits,
                labels
            )
        )


    # Same wide safe range as RETFound
    result = minimize_scalar(
        objective,
        bounds=(-5.0, 5.0),
        method="bounded",
        options={
            "xatol": 1e-8
        }
    )


    temperature = float(
        np.exp(
            result.x
        )
    )


    # T = 1 means no calibration.
    # Never accept a solution worse than T = 1.
    original_nll = (
        binary_nll_from_logits(
            logits,
            labels
        )
    )

    optimized_nll = (
        binary_nll_from_logits(
            logits / temperature,
            labels
        )
    )


    if optimized_nll > original_nll:

        temperature = 1.0


    return temperature


# ============================================================
# 6. FIT TEMPERATURE FOR EACH SEED
# ============================================================

results = {

    "model":
        "MedSigLIP",

    "method":
        "single_parameter_temperature_scaling",

    "objective":
        "binary_negative_log_likelihood",

    "fit_split":
        "prob_calibration",

    "ece_bins":
        ECE_BINS,

    "seeds":
        {}
}


print(
    "\n========== MEDSIGLIP TEMPERATURE SCALING =========="
)

print(
    "\nFitting data: prob_calibration ONLY"
)

print(
    "Risk calibration: NOT USED"
)

print(
    "Internal test: NOT USED"
)


for seed in SEEDS:

    # ========================================================
    # PROBABILITY-CALIBRATION SUBSET ONLY
    # ========================================================

    cal = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "prob_calibration"
        )
    ].copy()


    assert len(
        cal
    ) == 350


    logits = (
        cal["raw_logit"]
        .to_numpy(
            dtype=np.float64
        )
    )

    labels = (
        cal["binary_label"]
        .to_numpy(
            dtype=np.int64
        )
    )


    # ========================================================
    # BEFORE CALIBRATION
    # ========================================================

    raw_prob = expit(
        logits
    )


    nll_before = (
        binary_nll_from_logits(
            logits,
            labels
        )
    )


    brier_before = (
        brier_score(
            raw_prob,
            labels
        )
    )


    ece_before = (
        expected_calibration_error(
            raw_prob,
            labels,
            ECE_BINS
        )
    )


    # ========================================================
    # FIT TEMPERATURE
    # ========================================================

    temperature = (
        fit_temperature(
            logits,
            labels
        )
    )


    # ========================================================
    # AFTER CALIBRATION
    # ========================================================

    calibrated_logits = (
        logits
        /
        temperature
    )


    calibrated_prob = expit(
        calibrated_logits
    )


    nll_after = (
        binary_nll_from_logits(
            calibrated_logits,
            labels
        )
    )


    brier_after = (
        brier_score(
            calibrated_prob,
            labels
        )
    )


    ece_after = (
        expected_calibration_error(
            calibrated_prob,
            labels,
            ECE_BINS
        )
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    assert np.isfinite(
        temperature
    )

    assert temperature > 0

    assert (
        nll_after
        <=
        nll_before + 1e-10
    )


    # ========================================================
    # STORE
    # ========================================================

    results[
        "seeds"
    ][str(seed)] = {

        "temperature":
            float(
                temperature
            ),

        "calibration_samples":
            int(
                len(cal)
            ),

        "non_referable":
            int(
                (
                    labels == 0
                ).sum()
            ),

        "referable":
            int(
                (
                    labels == 1
                ).sum()
            ),

        "before": {

            "nll":
                float(
                    nll_before
                ),

            "brier":
                float(
                    brier_before
                ),

            "ece":
                float(
                    ece_before
                )
        },

        "after": {

            "nll":
                float(
                    nll_after
                ),

            "brier":
                float(
                    brier_after
                ),

            "ece":
                float(
                    ece_after
                )
        }
    }


    # ========================================================
    # REPORT
    # ========================================================

    print(
        f"\n---------- SEED {seed} ----------"
    )

    print(
        "Calibration samples:",
        len(cal)
    )

    print(
        "Temperature:",
        f"{temperature:.6f}"
    )

    print(
        "NLL:",
        f"{nll_before:.6f}",
        "->",
        f"{nll_after:.6f}"
    )

    print(
        "Brier:",
        f"{brier_before:.6f}",
        "->",
        f"{brier_after:.6f}"
    )

    print(
        "ECE:",
        f"{ece_before:.6f}",
        "->",
        f"{ece_after:.6f}"
    )


# ============================================================
# 7. SAVE ONE JSON
# ============================================================

with open(
    SAVE_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ============================================================
# 8. SUMMARY
# ============================================================

temperatures = np.array([
    results[
        "seeds"
    ][str(seed)][
        "temperature"
    ]
    for seed in SEEDS
])


print(
    "\n========== FINAL SUMMARY =========="
)

print(
    "Mean temperature:",
    f"{temperatures.mean():.6f}"
)

print(
    "Temperature SD:",
    f"{temperatures.std(ddof=1):.6f}"
)

print(
    "\nSaved permanently:"
)

print(
    SAVE_PATH
)

print(
    "\nRisk calibration was NOT used."
)

print(
    "Internal test was NOT used."
)

print(
    "\n========== COMPLETE =========="
)

========== INPUT CHECK ==========
Prediction rows: 17520
Seeds: [42, 43, 44, 45, 46]

========== MEDSIGLIP TEMPERATURE SCALING ==========

Fitting data: prob_calibration ONLY
Risk calibration: NOT USED
Internal test: NOT USED

---------- SEED 42 ----------
Calibration samples: 350
Temperature: 1.317355
NLL: 0.164383 -> 0.157691
Brier: 0.051544 -> 0.049789
ECE: 0.029429 -> 0.018389

---------- SEED 43 ----------
Calibration samples: 350
Temperature: 0.893643
NLL: 0.137747 -> 0.136850
Brier: 0.038739 -> 0.038257
ECE: 0.039346 -> 0.035125

---------- SEED 44 ----------
Calibration samples: 350
Temperature: 1.114815
NLL: 0.168290 -> 0.166983
Brier: 0.051757 -> 0.051181
ECE: 0.027108 -> 0.024158

---------- SEED 45 ----------
Calibration samples: 350
Temperature: 0.924790
NLL: 0.129434 -> 0.129001
Brier: 0.038008 -> 0.037845
ECE: 0.028557 -> 0.027238

---------- SEED 46 ----------
Calibration samples: 350
Temperature: 1.170520
NLL: 0.156290 -> 0.153824
Brier: 0.048694 -> 0.047724
ECE: 0.038

#**MEDSIGLIP FIXED-SEQUENCE LTT RISK CONTROL**

In [14]:
# ============================================================
# MEDSIGLIP FIXED-SEQUENCE LTT RISK CONTROL
#
# SAME PROCEDURE AS RETFOUND-GREEN
#
# Sequence/order:
#   DEVELOPMENT data ONLY
#
# Certification:
#   RISK_CALIBRATION data ONLY
#
# Targets:
#   alpha = 0.01, 0.02, 0.05
#   delta = 0.05
#
# No internal-test data used.
#
# Saves ONLY:
#   aptos_medsiglip_ltt_risk_control.csv
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import binom


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_PATH = (
    PROJECT_DIR /
    "aptos_medsiglip_raw_predictions.csv"
)

TEMP_PATH = (
    PROJECT_DIR /
    "aptos_medsiglip_temperature_scaling.json"
)

SAVE_PATH = (
    PROJECT_DIR /
    "aptos_medsiglip_ltt_risk_control.csv"
)


# ============================================================
# 2. EXACT SAME SETTINGS AS RETFOUND
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

ALPHAS = [
    0.01,
    0.02,
    0.05
]

DELTA = 0.05

N_CANDIDATES = 50

QUANTILES = np.linspace(
    0.02,
    0.98,
    N_CANDIDATES
)


# ============================================================
# 3. LOAD
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

with open(
    TEMP_PATH,
    "r"
) as f:
    temperature_data = json.load(f)


assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"].unique().tolist()
) == SEEDS

assert pred[
    "raw_logit"
].isna().sum() == 0

assert np.isfinite(
    pred["raw_logit"].values
).all()


# ============================================================
# 4. HOEFFDING-BENTKUS P-VALUE
# SAME AS RETFOUND
# ============================================================

def binary_kl(
    x,
    mu
):

    x = float(x)
    mu = float(mu)

    if x <= 0:
        return float(
            np.log(
                1.0 /
                (1.0 - mu)
            )
        )

    if x >= 1:
        return float(
            np.log(
                1.0 / mu
            )
        )

    return float(
        x * np.log(
            x / mu
        )
        +
        (1.0 - x)
        *
        np.log(
            (1.0 - x) /
            (1.0 - mu)
        )
    )


def hb_p_value(
    empirical_risk,
    n,
    alpha
):

    r_hat = float(
        empirical_risk
    )

    if r_hat >= alpha:
        return 1.0


    bentkus = (
        np.e
        *
        binom.cdf(
            int(
                np.ceil(
                    n * r_hat
                )
            ),
            n,
            alpha
        )
    )


    hoeffding = np.exp(
        -n
        *
        binary_kl(
            min(
                r_hat,
                alpha
            ),
            alpha
        )
    )


    return float(
        min(
            1.0,
            bentkus,
            hoeffding
        )
    )


# ============================================================
# 5. LTT LOSS
# ============================================================

def evaluate_threshold(
    probabilities,
    labels,
    threshold,
    alpha
):

    auto_clear = (
        probabilities
        <= threshold
    )

    auto_clear_count = int(
        auto_clear.sum()
    )


    errors = int(
        (
            auto_clear
            &
            (labels == 1)
        ).sum()
    )


    n = len(
        labels
    )


    coverage = (
        auto_clear_count
        /
        n
    )


    review_burden = (
        1.0
        -
        coverage
    )


    if auto_clear_count > 0:

        observed_risk = (
            errors
            /
            auto_clear_count
        )

    else:

        observed_risk = np.nan


    # --------------------------------------------------------
    # Exact same transformed conditional-risk loss
    #
    # L_t = A*y + alpha*(1-A)
    # --------------------------------------------------------

    A = auto_clear.astype(
        np.float64
    )


    losses = (
        A * labels
        +
        alpha * (
            1.0 - A
        )
    )


    empirical_ltt_loss = float(
        losses.mean()
    )


    return {

        "auto_clear_count":
            auto_clear_count,

        "errors":
            errors,

        "coverage":
            float(
                coverage
            ),

        "review_burden":
            float(
                review_burden
            ),

        "observed_risk":
            (
                float(
                    observed_risk
                )
                if np.isfinite(
                    observed_risk
                )
                else np.nan
            ),

        "empirical_ltt_loss":
            empirical_ltt_loss
    }


# ============================================================
# 6. RUN LTT
# ============================================================

final_rows = []


print(
    "========== MEDSIGLIP FIXED-SEQUENCE LTT =========="
)

print(
    "\nMethod:"
)

print(
    "Fixed-sequence Learn-Then-Test"
)

print(
    "P-value: Hoeffding-Bentkus"
)

print(
    "\nRisk targets:",
    ALPHAS
)

print(
    "Delta:",
    DELTA
)

print(
    "Candidate grid:",
    N_CANDIDATES
)

print(
    "\nInternal test used: NO"
)


for seed in SEEDS:

    # ========================================================
    # TEMPERATURE
    # ========================================================

    temperature = float(
        temperature_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    # ========================================================
    # DEVELOPMENT
    # ========================================================

    dev_df = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "development"
        )
    ].copy()


    # ========================================================
    # RISK CALIBRATION
    # ========================================================

    risk_df = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "risk_calibration"
        )
    ].copy()


    assert len(
        dev_df
    ) == 2102


    assert len(
        risk_df
    ) == 526


    # ========================================================
    # APPLY SOURCE TEMPERATURE
    # ========================================================

    dev_prob = expit(
        dev_df[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        temperature
    )


    risk_prob = expit(
        risk_df[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        temperature
    )


    dev_labels = (
        dev_df[
            "binary_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    risk_labels = (
        risk_df[
            "binary_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # ========================================================
    # CANDIDATE GRID
    # DEVELOPMENT ONLY
    # ========================================================

    candidate_thresholds = np.quantile(
        dev_prob,
        QUANTILES
    )


    candidate_thresholds = np.unique(
        candidate_thresholds
    )


    K = len(
        candidate_thresholds
    )


    print(
        f"\n\n========== SEED {seed} =========="
    )

    print(
        "Development samples:",
        len(dev_df)
    )

    print(
        "Risk-calibration samples:",
        len(risk_df)
    )

    print(
        "Temperature:",
        f"{temperature:.6f}"
    )

    print(
        "Candidate thresholds:",
        K
    )


    # ========================================================
    # EACH RISK TARGET
    # ========================================================

    for alpha in ALPHAS:

        # ----------------------------------------------------
        # A. FIX SEQUENCE USING DEVELOPMENT ONLY
        # ----------------------------------------------------

        ordering_rows = []


        for threshold in candidate_thresholds:

            dev_result = evaluate_threshold(
                dev_prob,
                dev_labels,
                threshold,
                alpha
            )


            ordering_rows.append({

                "threshold":
                    float(
                        threshold
                    ),

                "development_ltt_loss":
                    float(
                        dev_result[
                            "empirical_ltt_loss"
                        ]
                    ),

                "development_coverage":
                    float(
                        dev_result[
                            "coverage"
                        ]
                    )
            })


        ordering_df = pd.DataFrame(
            ordering_rows
        )


        # Same RETFound ordering:
        # 1. lowest development LTT loss
        # 2. if tied, highest development coverage
        ordering_df = (
            ordering_df
            .sort_values(
                [
                    "development_ltt_loss",
                    "development_coverage"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # B. FIXED-SEQUENCE TEST ON RISK CALIBRATION
        # ----------------------------------------------------

        certified_candidates = []

        stop_p_value = np.nan
        stop_threshold = np.nan


        for position, row in ordering_df.iterrows():

            threshold = float(
                row[
                    "threshold"
                ]
            )


            risk_result = evaluate_threshold(
                risk_prob,
                risk_labels,
                threshold,
                alpha
            )


            empirical_loss = float(
                risk_result[
                    "empirical_ltt_loss"
                ]
            )


            p_value = hb_p_value(
                empirical_risk=
                    empirical_loss,

                n=len(
                    risk_labels
                ),

                alpha=alpha
            )


            # ------------------------------------------------
            # certify and continue
            # ------------------------------------------------

            if (
                p_value <= DELTA
                and
                risk_result[
                    "auto_clear_count"
                ] > 0
            ):

                certified_candidates.append({

                    "sequence_position":
                        int(
                            position + 1
                        ),

                    "threshold":
                        threshold,

                    "p_value":
                        float(
                            p_value
                        ),

                    **risk_result
                })


            else:

                # first failure => STOP
                stop_p_value = float(
                    p_value
                )

                stop_threshold = float(
                    threshold
                )

                break


        # ====================================================
        # SELECT HIGHEST-COVERAGE CERTIFIED CANDIDATE
        # ====================================================

        if len(
            certified_candidates
        ) > 0:

            certified_df = pd.DataFrame(
                certified_candidates
            )


            selected = (
                certified_df
                .sort_values(
                    [
                        "coverage",
                        "threshold"
                    ],
                    ascending=[
                        False,
                        True
                    ]
                )
                .iloc[0]
            )


            certified = True


            selected_threshold = float(
                selected[
                    "threshold"
                ]
            )


            selected_p_value = float(
                selected[
                    "p_value"
                ]
            )


            auto_clear_count = int(
                selected[
                    "auto_clear_count"
                ]
            )


            errors = int(
                selected[
                    "errors"
                ]
            )


            observed_risk = float(
                selected[
                    "observed_risk"
                ]
            )


            coverage = float(
                selected[
                    "coverage"
                ]
            )


            review_burden = float(
                selected[
                    "review_burden"
                ]
            )


            empirical_ltt_loss = float(
                selected[
                    "empirical_ltt_loss"
                ]
            )


            sequence_position = int(
                selected[
                    "sequence_position"
                ]
            )


        else:

            certified = False

            selected_threshold = np.nan
            selected_p_value = np.nan

            auto_clear_count = 0
            errors = 0

            observed_risk = np.nan

            coverage = 0.0

            review_burden = 1.0

            empirical_ltt_loss = np.nan

            sequence_position = np.nan


        # ====================================================
        # SAVE SUMMARY ROW
        # ====================================================

        final_rows.append({

            "seed":
                int(
                    seed
                ),

            "alpha":
                float(
                    alpha
                ),

            "delta":
                float(
                    DELTA
                ),

            "method":
                "Fixed_Sequence_LTT_Hoeffding_Bentkus",

            "ordering_source":
                "development_only",

            "candidate_grid_source":
                "development_probability_quantiles",

            "candidate_count":
                int(
                    K
                ),

            "risk_calibration_n":
                int(
                    len(
                        risk_df
                    )
                ),

            "certified":
                bool(
                    certified
                ),

            "certified_candidates_before_stop":
                int(
                    len(
                        certified_candidates
                    )
                ),

            "selected_sequence_position":
                sequence_position,

            "selected_threshold":
                selected_threshold,

            "selected_p_value":
                selected_p_value,

            "auto_clear_count":
                int(
                    auto_clear_count
                ),

            "errors":
                int(
                    errors
                ),

            "observed_auto_clear_risk":
                observed_risk,

            "coverage":
                float(
                    coverage
                ),

            "review_burden":
                float(
                    review_burden
                ),

            "empirical_ltt_loss":
                empirical_ltt_loss,

            "first_failed_threshold":
                stop_threshold,

            "first_failed_p_value":
                stop_p_value
        })


        # ====================================================
        # REPORT
        # ====================================================

        print(
            f"\n--- Target alpha = {alpha:.2%} ---"
        )

        print(
            "Certified:",
            "YES"
            if certified
            else "NO"
        )

        print(
            "Candidates certified before stop:",
            len(
                certified_candidates
            )
        )


        if certified:

            print(
                "Selected threshold:",
                f"{selected_threshold:.6f}"
            )

            print(
                "Selected p-value:",
                f"{selected_p_value:.6f}"
            )

            print(
                "Automatic non-referral count:",
                auto_clear_count
            )

            print(
                "Referable errors:",
                errors
            )

            print(
                "Observed non-referral risk:",
                f"{observed_risk:.4f}"
            )

            print(
                "Coverage:",
                f"{coverage:.4f}"
            )

            print(
                "Review burden:",
                f"{review_burden:.4f}"
            )

        else:

            print(
                "No threshold certified."
            )


        if np.isfinite(
            stop_p_value
        ):

            print(
                "First failed p-value:",
                f"{stop_p_value:.6f}"
            )


# ============================================================
# 7. FINAL TABLE
# ============================================================

results = pd.DataFrame(
    final_rows
)

assert len(
    results
) == 15


# ============================================================
# 8. SAVE
# ============================================================

results.to_csv(
    SAVE_PATH,
    index=False
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print(
    "\n\n========== FINAL FIXED-SEQUENCE LTT SUMMARY =========="
)

print(
    "\nTotal seed x alpha combinations:",
    len(
        results
    )
)

print(
    "Certified combinations:",
    int(
        results[
            "certified"
        ].sum()
    )
)


for alpha in ALPHAS:

    subset = results[
        results[
            "alpha"
        ] == alpha
    ]


    certified_subset = subset[
        subset[
            "certified"
        ]
    ]


    print(
        f"\nTarget alpha = {alpha:.2%}"
    )

    print(
        "Certified seeds:",
        len(
            certified_subset
        ),
        "/ 5"
    )


    if len(
        certified_subset
    ) > 0:

        print(
            "Mean coverage:",
            f"{certified_subset['coverage'].mean():.4f}"
        )

        print(
            "Min coverage:",
            f"{certified_subset['coverage'].min():.4f}"
        )

        print(
            "Max coverage:",
            f"{certified_subset['coverage'].max():.4f}"
        )

        print(
            "Total observed referable errors:",
            int(
                certified_subset[
                    "errors"
                ].sum()
            )
        )

    else:

        print(
            "No seed certified this target."
        )


print(
    "\nDATA SEPARATION CHECK:"
)

print(
    "Development used for sequence construction: YES"
)

print(
    "Risk calibration used for certification: YES"
)

print(
    "Risk-calibration labels used for ordering: NO"
)

print(
    "Internal test used: NO"
)

print(
    "\nSaved final LTT result:"
)

print(
    SAVE_PATH
)

print(
    "\n========== COMPLETE =========="
)

========== MEDSIGLIP FIXED-SEQUENCE LTT ==========

Method:
Fixed-sequence Learn-Then-Test
P-value: Hoeffding-Bentkus

Risk targets: [0.01, 0.02, 0.05]
Delta: 0.05
Candidate grid: 50

Internal test used: NO


========== SEED 42 ==========
Development samples: 2102
Risk-calibration samples: 526
Temperature: 1.317355
Candidate thresholds: 50

--- Target alpha = 1.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.
First failed p-value: 0.940345

--- Target alpha = 2.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.
First failed p-value: 0.473711

--- Target alpha = 5.00% ---
Certified: YES
Candidates certified before stop: 2
Selected threshold: 0.227806
Selected p-value: 0.029188
Automatic non-referral count: 289
Referable errors: 3
Observed non-referral risk: 0.0104
Coverage: 0.5494
Review burden: 0.4506
First failed p-value: 0.145250


========== SEED 43 ==========
Development samples: 2102
Risk-calibration samples: 526
Temp

#**MEDSIGLIP CURRENT FINAL SUMMARY**

In [17]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import beta

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_FILE = (
    BASE /
    "aptos_medsiglip_raw_predictions.csv"
)

TEMP_FILE = (
    BASE /
    "aptos_medsiglip_temperature_scaling.json"
)

LTT_FILE = (
    BASE /
    "aptos_medsiglip_ltt_risk_control.csv"
)


# ============================================================
# 2. SETTINGS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

PRIMARY_ALPHA = 0.05

CLASSIFICATION_THRESHOLD = 0.50


# ============================================================
# 3. LOAD EXISTING RESULTS
# ============================================================

pred = pd.read_csv(
    PRED_FILE,
    dtype={"image_id": str}
)

with open(
    TEMP_FILE,
    "r"
) as f:

    temp_data = json.load(f)


ltt = pd.read_csv(
    LTT_FILE
)


assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"]
    .unique()
    .tolist()
) == SEEDS


# ============================================================
# 4. INTERNAL CLASSIFICATION EVALUATION
# ============================================================

classification_rows = []


for seed in SEEDS:

    test = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "internal_test"
        )
    ].copy()


    assert len(test) == 526


    T = float(
        temp_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    logits = test[
        "raw_logit"
    ].to_numpy(
        dtype=np.float64
    )


    labels = test[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )


    probability = expit(
        logits / T
    )


    predictions = (
        probability
        >= CLASSIFICATION_THRESHOLD
    ).astype(int)


    # --------------------------------------------
    # Confusion matrix
    # --------------------------------------------

    tn, fp, fn, tp = (
        confusion_matrix(
            labels,
            predictions,
            labels=[0, 1]
        )
        .ravel()
    )


    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )


    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )


    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )


    classification_rows.append({

        "seed":
            seed,

        "n_test":
            len(test),

        "AUROC":
            roc_auc_score(
                labels,
                probability
            ),

        "AUPRC":
            average_precision_score(
                labels,
                probability
            ),

        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "PPV":
            ppv,

        "NPV":
            npv,

        "F1":
            f1_score(
                labels,
                predictions
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                labels,
                predictions
            )
    })


classification_df = pd.DataFrame(
    classification_rows
)


# ============================================================
# 5. CALIBRATION SUMMARY
# ============================================================

calibration_rows = []


for seed in SEEDS:

    s = temp_data[
        "seeds"
    ][str(seed)]


    calibration_rows.append({

        "seed":
            seed,

        "temperature":
            s["temperature"],

        "NLL_before":
            s["before"]["nll"],

        "NLL_after":
            s["after"]["nll"],

        "Brier_before":
            s["before"]["brier"],

        "Brier_after":
            s["after"]["brier"],

        "ECE_before":
            s["before"]["ece"],

        "ECE_after":
            s["after"]["ece"]
    })


calibration_df = pd.DataFrame(
    calibration_rows
)


# ============================================================
# 6. PRIMARY 5% LTT POLICIES
# ============================================================

primary_ltt = ltt[
    np.isclose(
        ltt["alpha"],
        PRIMARY_ALPHA
    )
].copy()


assert len(primary_ltt) == 5


# ============================================================
# 7. HELD-OUT INTERNAL SELECTIVE-RISK EVALUATION
# ============================================================

risk_rows = []


for seed in SEEDS:

    test = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "internal_test"
        )
    ].copy()


    T = float(
        temp_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    probability = expit(
        test[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        T
    )


    labels = test[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )


    policy = primary_ltt[
        primary_ltt[
            "seed"
        ] == seed
    ].iloc[0]


    certified = bool(
        policy[
            "certified"
        ]
    )


    # ========================================================
    # FAIL-SAFE RULE
    #
    # If source LTT did not certify a policy,
    # automatic non-referral is disabled.
    # ========================================================

    if certified:

        threshold = float(
            policy[
                "selected_threshold"
            ]
        )


        auto_non_referral = (
            probability
            <= threshold
        )


        n_auto = int(
            auto_non_referral.sum()
        )


        errors = int(
            (
                auto_non_referral
                &
                (labels == 1)
            ).sum()
        )


        coverage = (
            n_auto
            /
            len(test)
        )


        review_burden = (
            1.0
            -
            coverage
        )


        if n_auto > 0:

            risk = (
                errors
                /
                n_auto
            )


            # --------------------------------------------
            # One-sided 95% exact binomial upper bound
            # Evaluation uncertainty only.
            # NOT the LTT certification guarantee.
            # --------------------------------------------

            if errors == n_auto:

                upper95 = 1.0

            else:

                upper95 = float(
                    beta.ppf(
                        0.95,
                        errors + 1,
                        n_auto - errors
                    )
                )

        else:

            risk = np.nan
            upper95 = np.nan


    else:

        threshold = np.nan

        n_auto = 0
        errors = 0

        coverage = 0.0

        review_burden = 1.0

        risk = np.nan
        upper95 = np.nan


    risk_rows.append({

        "seed":
            seed,

        "certified":
            certified,

        "threshold":
            threshold,

        "test_n":
            len(test),

        "automatic_non_referral_n":
            n_auto,

        "referable_errors":
            errors,

        "coverage":
            coverage,

        "non_referral_risk":
            risk,

        "upper_95":
            upper95,

        "review_burden":
            review_burden,

        "observed_target_met":
            (
                bool(
                    risk <= PRIMARY_ALPHA
                )
                if np.isfinite(risk)
                else False
            ),

        "upper_bound_below_target":
            (
                bool(
                    upper95 <= PRIMARY_ALPHA
                )
                if np.isfinite(upper95)
                else False
            )
    })


risk_df = pd.DataFrame(
    risk_rows
)


# ============================================================
# 8. PRINT COMPLETE RESULTS
# ============================================================

print(
    "=" * 78
)

print(
    "MEDSIGLIP COMPLETE SOURCE-DOMAIN RESULTS"
)

print(
    "=" * 78
)


# ============================================================
# CLASSIFICATION
# ============================================================

print(
    "\n1. HELD-OUT INTERNAL CLASSIFICATION"
)

print(
    "-" * 78
)


print(
    classification_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nMEAN ± SD"
)


metric_names = [
    "AUROC",
    "AUPRC",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
    "Balanced_Accuracy"
]


for metric in metric_names:

    print(
        f"{metric:<20}"
        f"{classification_df[metric].mean():.4f}"
        f" ± "
        f"{classification_df[metric].std(ddof=1):.4f}"
    )


# ============================================================
# CALIBRATION
# ============================================================

print(
    "\n\n2. PROBABILITY CALIBRATION"
)

print(
    "-" * 78
)


print(
    calibration_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nMEAN ± SD"
)


print(
    "NLL    "
    f"{calibration_df['NLL_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['NLL_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['NLL_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['NLL_after'].std(ddof=1):.4f}"
)


print(
    "Brier  "
    f"{calibration_df['Brier_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['Brier_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['Brier_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['Brier_after'].std(ddof=1):.4f}"
)


print(
    "ECE    "
    f"{calibration_df['ECE_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['ECE_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['ECE_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['ECE_after'].std(ddof=1):.4f}"
)


# ============================================================
# LTT CERTIFICATION
# ============================================================

print(
    "\n\n3. SOURCE LTT CERTIFICATION"
)

print(
    "-" * 78
)


for alpha in [
    0.01,
    0.02,
    0.05
]:

    sub = ltt[
        np.isclose(
            ltt["alpha"],
            alpha
        )
    ]

    print(
        f"{100*alpha:.0f}% target: "
        f"{int(sub['certified'].sum())}/5 seeds certified"
    )


# ============================================================
# HELD-OUT RISK RESULT
# ============================================================

print(
    "\n\n4. HELD-OUT INTERNAL AUTOMATIC NON-REFERRAL"
)

print(
    "-" * 78
)


print(
    risk_df[
        [
            "seed",
            "certified",
            "threshold",
            "automatic_non_referral_n",
            "referable_errors",
            "coverage",
            "non_referral_risk",
            "upper_95",
            "review_burden",
            "observed_target_met",
            "upper_bound_below_target"
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


# ============================================================
# 9. FINAL PAPER-LEVEL SUMMARY
# ============================================================

print(
    "\n\n5. FINAL MEDSIGLIP SUMMARY"
)

print(
    "=" * 78
)


print(
    "\nCLASSIFICATION"
)


for metric in metric_names:

    print(
        f"{metric:<20}"
        f"{classification_df[metric].mean():.4f}"
        f" ± "
        f"{classification_df[metric].std(ddof=1):.4f}"
    )


print(
    "\nCALIBRATION"
)


print(
    f"NLL:   "
    f"{calibration_df['NLL_before'].mean():.4f}"
    " -> "
    f"{calibration_df['NLL_after'].mean():.4f}"
)


print(
    f"Brier: "
    f"{calibration_df['Brier_before'].mean():.4f}"
    " -> "
    f"{calibration_df['Brier_after'].mean():.4f}"
)


print(
    f"ECE:   "
    f"{calibration_df['ECE_before'].mean():.4f}"
    " -> "
    f"{calibration_df['ECE_after'].mean():.4f}"
)


print(
    "\n5% SOURCE RISK CONTROL"
)


print(
    "LTT certification:"
    f" {int(risk_df['certified'].sum())}/5 seeds"
)


print(
    "Operational automatic non-referral coverage:"
    f" {100 * risk_df['coverage'].mean():.2f}"
    f" ± "
    f"{100 * risk_df['coverage'].std(ddof=1):.2f}%"
)


print(
    "Operational review burden:"
    f" {100 * risk_df['review_burden'].mean():.2f}"
    f" ± "
    f"{100 * risk_df['review_burden'].std(ddof=1):.2f}%"
)


# Risk/upper bound are meaningful only for seeds
# where a policy was actually certified and evaluated.

evaluated_risk = risk_df[
    risk_df[
        "certified"
    ]
].copy()


if len(
    evaluated_risk
) > 0:

    print(
        "Observed non-referral risk among certified runs:"
        f" {100 * evaluated_risk['non_referral_risk'].mean():.2f}"
        f" ± "
        f"{100 * evaluated_risk['non_referral_risk'].std(ddof=1):.2f}%"
    )


    print(
        "One-sided 95% upper risk bound:"
        f" {100 * evaluated_risk['upper_95'].mean():.2f}"
        f" ± "
        f"{100 * evaluated_risk['upper_95'].std(ddof=1):.2f}%"
    )


    print(
        "5% observed-risk target satisfied:"
        f" {int(evaluated_risk['observed_target_met'].sum())}"
        f"/{len(evaluated_risk)} evaluated certified runs"
    )


    print(
        "95% upper bound below 5%:"
        f" {int(evaluated_risk['upper_bound_below_target'].sum())}"
        f"/{len(evaluated_risk)} evaluated certified runs"
    )


    print(
        "Total held-out referable errors:"
        f" {int(evaluated_risk['referable_errors'].sum())}"
    )


print(
    "\nSeed(s) with no certified 5% source policy:"
)


uncertified = risk_df[
    ~risk_df[
        "certified"
    ]
]["seed"].tolist()


print(
    uncertified
    if len(uncertified) > 0
    else "None"
)


print(
    "\nNo files created or modified by this cell."
)

print(
    "=" * 78
)

MEDSIGLIP COMPLETE SOURCE-DOMAIN RESULTS

1. HELD-OUT INTERNAL CLASSIFICATION
------------------------------------------------------------------------------
 seed  n_test  AUROC  AUPRC  Sensitivity  Specificity    PPV    NPV     F1  Balanced_Accuracy
   42     526 0.9845 0.9724       0.9707       0.9252 0.8924 0.9802 0.9299             0.9480
   43     526 0.9824 0.9695       0.8732       0.9502 0.9179 0.9215 0.8950             0.9117
   44     526 0.9796 0.9619       0.9512       0.9252 0.8904 0.9674 0.9198             0.9382
   45     526 0.9824 0.9651       0.9660       0.9156 0.8805 0.9767 0.9213             0.9408
   46     526 0.9823 0.9696       0.9515       0.9125 0.8750 0.9669 0.9116             0.9320

MEAN ± SD
AUROC               0.9822 ± 0.0018
AUPRC               0.9677 ± 0.0042
Sensitivity         0.9425 ± 0.0397
Specificity         0.9257 ± 0.0148
PPV                 0.8913 ± 0.0165
NPV                 0.9625 ± 0.0237
F1                  0.9155 ± 0.0132
Balanced_Accurac

| Result                     |    RETFound-Green |           MedSigLIP | Better         |
| -------------------------- | ----------------: | ------------------: | -------------- |
| AUROC                      |   0.9800 ± 0.0018 | **0.9822 ± 0.0018** | MedSigLIP      |
| AUPRC                      |   0.9648 ± 0.0051 | **0.9677 ± 0.0042** | MedSigLIP      |
| Sensitivity                |            0.9211 |          **0.9425** | MedSigLIP      |
| Specificity                |        **0.9276** |              0.9257 | RETFound, tiny |
| F1                         |            0.9057 |          **0.9155** | MedSigLIP      |
| Balanced accuracy          |            0.9244 |          **0.9341** | MedSigLIP      |
| ECE after calibration      |            0.0293 |          **0.0277** | MedSigLIP      |
| 5% LTT certification       |           **5/5** |                 4/5 | **RETFound**   |
| Operational coverage       | **51.06 ± 1.61%** |      43.76 ± 24.49% | **RETFound**   |
| Held-out non-referral risk |  **0.37 ± 0.36%** |       0.95 ± 0.57%* | **RETFound**   |
| 95% upper risk bound       |  **1.71 ± 0.57%** |       2.51 ± 0.79%* | **RETFound**   |
